In [ ]:
# ================== INICIALIZACIÓN COMPLETA DEL ENTORNO ==================
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import networkx as nx
import time
import requests

# Funciones de optimización TSP
def obtener_matriz_distancias_osrm(puntos):
    n = len(puntos)
    matriz = [[0]*n for _ in range(n)]
    url_base = 'http://router.project-osrm.org/route/v1/driving/'
    for i in range(n):
        for j in range(n):
            if i == j:
                matriz[i][j] = 0
            else:
                coords = f"{puntos[i][0]},{puntos[i][1]};{puntos[j][0]},{puntos[j][1]}"
                url = url_base + coords + '?overview=false'
                try:
                    r = requests.get(url)
                    data = r.json()
                    matriz[i][j] = data['routes'][0]['distance']/1000 if 'routes' in data and data['routes'] else float('inf')
                except:
                    matriz[i][j] = float('inf')
                time.sleep(0.1)
    return matriz

def resolver_tsp(matriz):
    n = len(matriz)
    G = nx.Graph()
    for i in range(n):
        for j in range(n):
            if i != j:
                G.add_edge(i, j, weight=matriz[i][j])
    ciclo = nx.approximation.traveling_salesman_problem(G, weight='weight', cycle=True)
    distancia_total = sum(matriz[ciclo[i]][ciclo[i+1]] for i in range(len(ciclo)-1))
    return ciclo, distancia_total

# Carga de datos y widgets
ARCHIVO_DATOS = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Clientes_Ruta.xlsx'
df = pd.read_excel(ARCHIVO_DATOS, sheet_name=None)
df_clientes = df[list(df.keys())[0]]
df_cdr = df[list(df.keys())[1]]

# Mostrar cantidad de clientes por ruta
rutas_disponibles = sorted(df_clientes['Ruta'].dropna().unique())
clientes_por_ruta = {ruta: df_clientes[df_clientes['Ruta'] == ruta].shape[0] for ruta in rutas_disponibles}
rutas_labels = [f"{ruta} ({clientes_por_ruta[ruta]} clientes)" for ruta in rutas_disponibles]

# Widget de selección de rutas
rutas_widget = widgets.SelectMultiple(options=list(zip(rutas_labels, rutas_disponibles)), value=tuple(rutas_disponibles), description='Rutas a optimizar:', style={'description_width': 'initial'}, layout=widgets.Layout(width='60%'))

# Widget de selección de METs con casillas
mets_disponibles = sorted(df_cdr['CodMET'].dropna().unique())
met_checkboxes = [widgets.Checkbox(value=False, description=str(met), indent=False) for met in mets_disponibles]
met_box = widgets.VBox([widgets.Label('Selecciona uno o varios METs:')] + met_checkboxes)

# Widget para definir cantidad de clientes a procesar
clientes_a_procesar = widgets.IntText(value=10, description='Cantidad de clientes a procesar:', style={'description_width': 'initial'}, layout=widgets.Layout(width='30%'))

select_all_btn = widgets.Button(description='Seleccionar todo', button_style='info')
deselect_all_btn = widgets.Button(description='Deseleccionar todo', button_style='warning')
param_button = widgets.Button(description='Aplicar selección', button_style='success')
output_param = widgets.Output()

def seleccionar_todo(b):
    rutas_widget.value = tuple(rutas_disponibles)

def deseleccionar_todo(b):
    rutas_widget.value = tuple()

def aplicar_parametros(b):
    with output_param:
        clear_output()
        mets_seleccionados = [met.description for met in met_checkboxes if met.value]
        print(f'MET(s) seleccionados: {mets_seleccionados}')
        rutas_seleccionadas = list(rutas_widget.value)
        print(f'Rutas a optimizar: {rutas_seleccionadas}')
        print(f'Cantidad de clientes a procesar: {clientes_a_procesar.value}')

select_all_btn.on_click(seleccionar_todo)
deselect_all_btn.on_click(deseleccionar_todo)
param_button.on_click(aplicar_parametros)

# Mostrar el display interactivo
display(widgets.VBox([met_box, rutas_widget, clientes_a_procesar, widgets.HBox([select_all_btn, deselect_all_btn]), param_button, output_param]))
# Los parámetros seleccionados estarán en met_checkboxes, rutas_widget.value y clientes_a_procesar.value para usarse en la optimización.

In [ ]:
# ================== EXPORTAR RECORRIDOS ÓPTIMOS POR RUTA EN UN SOLO MAPA Y EXCEL POR MET CON RESÚMENES Y FORMATO PROFESIONAL ==================
import openpyxl
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter
from openpyxl.drawing.image import Image as XLImage
import os
from folium.features import CustomIcon, DivIcon
from datetime import datetime
from branca.element import Template, MacroElement
import folium
import random

output_dir = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Outputs'
os.makedirs(output_dir, exist_ok=True)
icon_met_path = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\95_24.png'
mets_seleccionados = [met.description for met in met_checkboxes if met.value]
rutas_seleccionadas = list(rutas_widget.value)
num_clientes = clientes_a_procesar.value
fecha_hora = datetime.now().strftime('%Y%m%d_%H%M%S')

for met in mets_seleccionados:
    met_row = df_cdr[df_cdr['CodMET'] == met]
    if met_row.empty:
        print(f'No se encontró información para el MET {met}.')
        continue
    met_row = met_row.iloc[0]
    mapa = folium.Map(location=[met_row['U_latitud'], met_row['U_longitud']], zoom_start=10, tiles='OpenStreetMap')
    colores = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'white', 'pink', 'lightblue', 'lightgreen', 'gray', 'black', 'lightgray']
    export_rows = []
    resumen_rutas = []
    distancia_total_met = 0
    total_clientes_met = 0
    resumen_maximos = []
    for idx_ruta, ruta in enumerate(rutas_seleccionadas):
        clientes_ruta = df_clientes[df_clientes['Ruta'] == ruta].head(num_clientes)
        if clientes_ruta.empty:
            print(f'No hay clientes para la ruta {ruta}.')
            continue
        puntos = [(met_row['U_longitud'], met_row['U_latitud'])] + [(row['U_longitud'], row['U_latitud']) for _, row in clientes_ruta.iterrows()]
        codigos = [met] + list(clientes_ruta['CodCli'])
        matriz = obtener_matriz_distancias_osrm(puntos)
        ciclo, distancia_total = resolver_tsp(matriz)
        ciclo_filtrado = []
        vistos = set()
        for idx, val in enumerate(ciclo):
            if idx == 0 or idx == len(ciclo)-1 or val not in vistos:
                ciclo_filtrado.append(val)
                vistos.add(val)
        if ciclo_filtrado[-1] != 0:
            ciclo_filtrado.append(0)
        recorrido_codigos = [codigos[i] for i in ciclo_filtrado]

        distancias = []
        max_dist = 0
        max_from = ''
        max_to = ''
        for i in range(len(ciclo_filtrado)-1):
            d = matriz[ciclo_filtrado[i]][ciclo_filtrado[i+1]]
            distancias.append(d)
            if d > max_dist:
                max_dist = d
                max_from = codigos[ciclo_filtrado[i]]
                max_to = codigos[ciclo_filtrado[i+1]]

        # Resumen por ruta
        total_clientes_ruta = len(clientes_ruta)
        distancia_promedio_ruta = distancia_total / total_clientes_ruta if total_clientes_ruta > 0 else 0
        resumen_rutas.append({
            'Ruta': ruta,
            'Clientes': total_clientes_ruta,
            'Distancia_total_km': distancia_total,
            'Distancia_promedio_cliente_km': distancia_promedio_ruta,
            'Distancia_maxima_km': max_dist,
            'De': max_from,
            'A': max_to
        })
        distancia_total_met += distancia_total
        total_clientes_met += total_clientes_ruta

        # Exportar filas para Excel, agregando columna de ruta
        for idx, sec in enumerate(ciclo_filtrado):
            distancia_val = distancias[idx] if idx < len(distancias) else ''
            if sec == 0:
                export_rows.append({
                    'MET': met,
                    'Ruta': ruta,
                    'Secuencia': idx+1,
                    'Tipo': 'MET',
                    'Codigo': met,
                    'Longitud': met_row['U_longitud'],
                    'Latitud': met_row['U_latitud'],
                    'Nombre': met_row.get('Nombre', ''),
                    'Distancia_al_siguiente_km': distancia_val,
                    'Recorrido': ' -> '.join([str(x) for x in recorrido_codigos]),
                    'Distancia_total_km': distancia_total
                })
            else:
                cli_row = clientes_ruta.iloc[sec-1]
                export_rows.append({
                    'MET': met,
                    'Ruta': ruta,
                    'Secuencia': idx+1,
                    'Tipo': 'Cliente',
                    'Codigo': cli_row['CodCli'],
                    'Longitud': cli_row['U_longitud'],
                    'Latitud': cli_row['U_latitud'],
                    'Nombre': cli_row.get('Nombre', ''),
                    'Distancia_al_siguiente_km': distancia_val,
                    'Recorrido': ' -> '.join([str(x) for x in recorrido_codigos]),
                    'Distancia_total_km': distancia_total
                })

        # Dibujar recorrido en el mapa con color y tooltip por ruta
        ruta_coords = [puntos[i][::-1] for i in ciclo_filtrado]
        color_ruta = colores[idx_ruta % len(colores)]
        folium.PolyLine(ruta_coords, color=color_ruta, weight=5, opacity=0.8, tooltip=f"Recorrido óptimo MET {met} - Ruta {ruta} | Distancia: {distancia_total:.2f} km | Promedio por cliente: {distancia_promedio_ruta:.2f} km").add_to(mapa)

        # Marcadores de clientes y secuencia
        secuencia_cliente = 1
        for idx, sec in enumerate(ciclo_filtrado[1:]):
            if sec == 0:
                continue  # Saltar MET
            cli_row = clientes_ruta.iloc[sec-1]
            codcli = cli_row['CodCli']
            nombre = cli_row.get('Nombre', '')
            ruta_cliente = cli_row.get('Ruta', '')
            idx_ciclo = idx + 1
            distancia_anterior = matriz[ciclo_filtrado[idx_ciclo-1]][ciclo_filtrado[idx_ciclo]] if idx_ciclo > 0 else 'N/A'
            distancia_siguiente = matriz[ciclo_filtrado[idx_ciclo]][ciclo_filtrado[idx_ciclo+1]] if idx_ciclo+1 < len(ciclo_filtrado) else 'N/A'

            popup_html = f'''
            <div style=\"background: #fff; border-radius: 16px; box-shadow: 0 2px 8px rgba(0,0,0,0.15); padding: 14px; border: 2px solid #8BC34A; min-width: 260px; max-width: 340px;\">
                <div style=\"font-size:20px; color:{color_ruta}; font-weight:bold; margin-bottom:4px;\">
                    <span style=\"vertical-align:middle;\">👨‍💼</span> Cliente: <span style=\"color:{color_ruta};\">{codcli}</span>
                </div>
                <div style=\"font-size:16px; color:#333; margin-bottom:4px;\">
                    <span style=\"vertical-align:middle;\">📚</span> <b>Nombre:</b> {nombre}
                </div>
                <div style=\"font-size:15px; color:#2A81CB; margin-bottom:4px;\">
                    <span style=\"vertical-align:middle;\">🗺️</span> <b>Ruta:</b> {ruta_cliente}
                </div>
                <div style=\"font-size:15px; color:#CB2A2A; margin-bottom:4px;\">
                    <span style=\"vertical-align:middle;\">↩️</span> <b>Distancia anterior:</b> {distancia_anterior:.2f} km
                </div>
                <div style=\"font-size:15px; color:#2A81CB; margin-bottom:4px;\">
                    <span style=\"vertical-align:middle;\">➡️</span> <b>Distancia siguiente:</b> {distancia_siguiente:.2f} km
                </div>
                <div style=\"font-size:15px; color:#FFC107; margin-bottom:4px;\">
                    <span style=\"vertical-align:middle;\">🔢</span> <b>Número de secuencia:</b> {secuencia_cliente}
                </div>
            </div>
            '''
            folium.Marker(
                location=[cli_row['U_latitud'], cli_row['U_longitud']],
                popup=folium.Popup(popup_html, max_width=340),
                icon=folium.Icon(color=color_ruta, icon='info-sign')
            ).add_to(mapa)
            folium.Marker(
                location=[cli_row['U_latitud'], cli_row['U_longitud']],
                icon=DivIcon(
                    icon_size=(24,24),
                    icon_anchor=(12,12),
                    html=f'<div style="font-size:16px; color:white; background:{color_ruta}; border-radius:50%; width:24px; height:24px; text-align:center; line-height:24px; border:2px solid #fff;">{secuencia_cliente}</div>'
                ),
            ).add_to(mapa)
            secuencia_cliente += 1

    # MET marker
    folium.Marker(location=[met_row['U_latitud'], met_row['U_longitud']],
                  popup=folium.Popup(f'''<div style="background:#2A81CB; color:#fff; border-radius:14px; box-shadow:0 2px 8px rgba(0,0,0,0.15); padding:14px; border:2px solid #fff; min-width:220px; text-align:center;">
                    <div style="font-size:20px; font-weight:bold; margin-bottom:6px;">🏠 MET: <span style="color:#FFD600;">{met}</span></div>
                    <div style="font-size:15px; color:#fff; margin-bottom:2px;">Coordenadas:<br><span style="font-size:14px; color:#FFD600;">{met_row['U_latitud']}, {met_row['U_longitud']}</span></div>
                  </div>''', max_width=260),
                  icon=CustomIcon(icon_met_path, icon_size=(32,32))).add_to(mapa)

    # Título y resumen general
    distancia_promedio_met = distancia_total_met / total_clientes_met if total_clientes_met > 0 else 0
    # Calcular distancia máxima general y de dónde a dónde
    max_dist_general = 0
    max_from_general = ''
    max_to_general = ''
    for resumen in resumen_rutas:
        if resumen['Distancia_maxima_km'] > max_dist_general:
            max_dist_general = resumen['Distancia_maxima_km']
            max_from_general = resumen['De']
            max_to_general = resumen['A']

    resumen_html = (
        f'<div style="position: fixed; top: 130px; right: 35px; width: 320px; background-color: white; border: 2px solid #333; z-index: 9997; font-size: 11px; padding: 14px; border-radius: 12px; box-shadow: 0 2px 8px rgba(0,0,0,0.3);">'
        f'<h3 style="margin-top: 0; color: #333; text-align: left; border-bottom: 2px solid #ddd; padding-bottom: 5px; font-size:14px;"><span style="font-size:14px;">📊 RESUMEN GENERAL MET {met}</span></h3>'
        f'<p style="margin: 5px 0; font-weight: bold; color: #333; font-size:12px;">🧮 Total clientes: {total_clientes_met}</p>'
        f'<p style="margin: 6px 0 4px 0; color: #444; font-size:11px;">🚗 <b>Distancia total:</b> {distancia_total_met:.2f} km</p>'
        f'<p style="margin: 6px 0 4px 0; color: #444; font-size:11px;">📏 <b>Distancia promedio por cliente:</b> {distancia_promedio_met:.2f} km</p>'
        f'<p style="margin: 6px 0 4px 0; color: #CB2A2A; font-size:11px;">🏆 <b>Distancia máxima:</b> {max_dist_general:.2f} km <b>De:</b> {max_from_general} <b>A:</b> {max_to_general}</p>'
        f'<hr style="margin:8px 0;"><span style="font-size:13px;">🔎 <b>Resumen por ruta:</b></span>'
    )
    for idx_ruta, resumen in enumerate(resumen_rutas):
        color_ruta = colores[idx_ruta % len(colores)]
        resumen_html += (
            f'<p style="margin: 4px 0 0 0; color: {color_ruta}; font-size:11px;">🛣️ <b>Ruta {resumen["Ruta"]}:</b> {resumen["Clientes"]} clientes, Distancia: {resumen["Distancia_total_km"]:.2f} km, Promedio por cliente: {resumen["Distancia_promedio_cliente_km"]:.2f} km, <span style="color:#CB2A2A;">🏆 Máxima: {resumen["Distancia_maxima_km"]:.2f} km de {resumen["De"]} a {resumen["A"]}</span></p>'
        )
    resumen_html += "</div>"
    mapa.get_root().html.add_child(folium.Element(resumen_html))

    titulo_html = f'''<div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); z-index: 9998; background-color: white; padding: 7px 20px; border: 2px solid #333; border-radius: 12px; box-shadow: 0 2px 8px rgba(0,0,0,0.3);"><h2 style="margin: 0; color: #333; text-align: center; font-size:14px;"><span style='font-size:14px;'>🗺️ MAPA RECORRIDOS ÓPTIMOS POR MET {met}</span></h2><p style="margin: 4px 0 0 0; text-align: center; color: #666; font-size: 9px;">Rutas analizadas: <b>{len(rutas_seleccionadas)}</b></p></div>'''
    mapa.get_root().html.add_child(folium.Element(titulo_html))

    nombre_mapa = os.path.join(output_dir, f'mapa_recorridos_optimos_{met}_{fecha_hora}.html')
    mapa.save(nombre_mapa)
    print(f'Mapa grupal exportado a: {nombre_mapa}')

    # Exportar Excel con formato profesional
    df_export = pd.DataFrame(export_rows)
    df_resumen = pd.DataFrame(resumen_rutas)
    resumen_general = pd.DataFrame([{
        'MET': met,
        'Total_clientes': total_clientes_met,
        'Distancia_total_km': distancia_total_met,
        'Distancia_promedio_cliente_km': distancia_promedio_met,
        'Distancia_maxima_km': max_dist_general,
        'De': max_from_general,
        'A': max_to_general
    }])
    excel_path = os.path.join(output_dir, f'recorridos_optimos_{met}_{fecha_hora}.xlsx')
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        df_export.to_excel(writer, sheet_name='Recorridos', index=False)
        df_resumen.to_excel(writer, sheet_name='Resumen por ruta', index=False)
        resumen_general.to_excel(writer, sheet_name='Resumen general', index=False)

    # Formatear el Excel con openpyxl
    wb = openpyxl.load_workbook(excel_path)
    # Estilos
    header_font = Font(bold=True, color='FFFFFF', size=12)
    header_fill = PatternFill('solid', fgColor='4F81BD')
    cell_fill = PatternFill('solid', fgColor='DDEBF7')
    border = Border(left=Side(style='thin', color='4F81BD'), right=Side(style='thin', color='4F81BD'), top=Side(style='thin', color='4F81BD'), bottom=Side(style='thin', color='4F81BD'))
    align = Alignment(horizontal='center', vertical='center', wrap_text=True)

    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        # Encabezados con emojis
        for col_idx, cell in enumerate(ws[1], 1):
            cell.font = header_font
            cell.fill = header_fill
            cell.alignment = align
            cell.border = border
            # Agregar emoji según columna
            if 'Ruta' in cell.value: cell.value = '🛣️ ' + str(cell.value)
            if 'Cliente' in cell.value: cell.value = '👨‍💼 ' + str(cell.value)
            if 'Distancia' in cell.value: cell.value = '📏 ' + str(cell.value)
            if 'Secuencia' in cell.value: cell.value = '🔢 ' + str(cell.value)
            if 'MET' in cell.value: cell.value = '🏠 ' + str(cell.value)
            if 'Nombre' in cell.value: cell.value = '📚 ' + str(cell.value)
            if 'De' == cell.value: cell.value = '🔴 De'
            if 'A' == cell.value: cell.value = '🟢 A'

        # Formato de celdas
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.fill = cell_fill
                cell.alignment = align
                cell.border = border

        # Ajustar ancho de columnas
        for col in ws.columns:
            max_length = 0
            col_letter = get_column_letter(col[0].column)
            for cell in col:
                try:
                    if cell.value:
                        max_length = max(max_length, len(str(cell.value)))
                except:
                    pass
            ws.column_dimensions[col_letter].width = max(12, min(max_length + 2, 40))

        # Fila resumen con color especial (solo para resumen general y por ruta)
        if sheet_name in ['Resumen general', 'Resumen por ruta']:
            for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
                for cell in row:
                    cell.fill = PatternFill('solid', fgColor='FFD966')

    wb.save(excel_path)
    print(f'Excel grupal exportado a: {excel_path}')